# 02 - Prepare WGS inputs

Same preparation as `01_prep_inputs_nba.ipynb`, for the samples that have
whole-genome sequencing but **no** NBA array data. Those samples are a separate
dataset in the meta-analysis; samples with both modalities stay in the array
run only, so that no one contributes twice.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

import gwas_config as cfg
import gwas_prep as prep

pd.options.mode.chained_assignment = None

cfg.describe()

In [ ]:
# Mount the release and working buckets with gcsfuse (run once per VM).
for cmd in cfg.mount_commands():
    print(cmd)

## Parameters

In [ ]:
ANALYSIS = 'CS_disc'   # 'CS_disc' or 'SURV'
ANCESTRY = 'EUR'
WGS_CALLER = 'dragen_joint_calling'   # joint-calling set to read genotypes from

OUT_DIR = cfg.prep_path(*cfg.prep_parts(ancestry=ANCESTRY, wgs=True), ANALYSIS)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUT_DIR)

## Clinical data and master key

In [ ]:
clinical = pd.read_csv(cfg.clinical_path(ANALYSIS), low_memory=False)
clinical = clinical.rename(columns={'id': 'GP2ID'}).drop_duplicates()

master = pd.read_csv(cfg.master_key())

in_master = master[master.GP2ID.isin(clinical.GP2ID)]
display(in_master.wgs_label.value_counts())
print(f'{len(in_master)} of {clinical.GP2ID.nunique()} samples found in the master key')

In [ ]:
# Samples sequenced but not genotyped on the array: these are the ones this
# notebook prepares.
wgs_only = in_master[(in_master.wgs == 1)
                     & ((in_master.nba == 0) | (in_master.nba.isnull()))]
print(f'{len(wgs_only)} WGS-only samples')
display(wgs_only.wgs_label.value_counts())

## Select the ancestry

In [ ]:
key_cols = ['GP2ID', 'study_type', 'GP2_phenotype', 'wgs_label']
full_data = clinical.merge(master[key_cols], on='GP2ID', how='inner')

subset = full_data[(full_data.GP2ID.isin(wgs_only.GP2ID))
                   & (full_data.wgs_label == ANCESTRY)]

display(subset.study_type.value_counts())
subset = subset[subset.study_type == 'Case(/Control)']

subset = prep.check_required_fields(subset, ANALYSIS)
print(f'{subset.GP2ID.nunique()} samples in {ANCESTRY}')
display(subset.cohort.value_counts())

## Keep-sample list and PLINK IDs

Belt and braces: the array covariate file for the same ancestry is excluded
explicitly, so a sample that slipped through the `nba` flag still cannot be
counted twice.

In [ ]:
psam = pd.read_csv(f'{cfg.wgs_pfile(ANCESTRY, 22, WGS_CALLER)}.psam', sep='\t')

nba_covar_path = cfg.prep_path(*cfg.prep_parts(ancestry=ANCESTRY), ANALYSIS,
                               cfg.covar_file(ANALYSIS, ANCESTRY))
already_run = set()
if nba_covar_path.exists():
    already_run = set(pd.read_csv(nba_covar_path, sep='\t').IID)
    print(f'{len(already_run)} samples already in the {ANCESTRY} array run')
else:
    print(f'no array covariate file at {nba_covar_path} - run notebook 01 first')

keep = psam[psam.IID.isin(subset.GP2ID) & ~psam.IID.isin(already_run)]
print(f'{len(keep)} samples have WGS genotypes and are not in the array run')

keep[['#FID', 'IID']].to_csv(OUT_DIR / f'{ANCESTRY}_keep_samples.txt',
                             sep='\t', index=False)

subset = subset.merge(keep[['#FID', 'IID']], left_on='GP2ID', right_on='IID', how='inner')
display(subset.head())

## Covariates and phenotypes

In [ ]:
covar = prep.format_covariates(subset, ANALYSIS)
covar.to_csv(OUT_DIR / cfg.covar_file(ANALYSIS, ANCESTRY), sep='\t', index=False)

print(f'{len(covar)} covariate rows')
display(covar.head())

bad_dd = covar[covar.disease_duration < 0]
if len(bad_dd):
    print(f'WARNING: {len(bad_dd)} samples have a negative disease duration')
    display(bad_dd)

In [ ]:
pheno = prep.format_phenotypes(subset, ANALYSIS)
pheno.to_csv(OUT_DIR / cfg.pheno_file(ANALYSIS, ANCESTRY), sep='\t', index=False)
display(pheno.head())

## Survival-only fixes

In [ ]:
if ANALYSIS == 'SURV':
    covar_path = OUT_DIR / cfg.covar_file(ANALYSIS, ANCESTRY)
    pheno_path = OUT_DIR / cfg.pheno_file(ANALYSIS, ANCESTRY)

    for path in (covar_path, pheno_path):
        backup = path.with_name(path.stem + '_PRE_FIX.tsv')
        backup.write_bytes(path.read_bytes())
        print(f'backed up {backup.name}')

    covar = pd.read_csv(covar_path, sep='\t')
    pheno = pd.read_csv(pheno_path, sep='\t')

    covar, pheno = prep.drop_zero_time_events(covar, pheno)
    covar, pheno = prep.drop_small_cohorts(covar, pheno, min_n=50)

    covar.to_csv(covar_path, sep='\t', index=False)
    pheno.to_csv(pheno_path, sep='\t', index=False)

    display(covar.cohort.value_counts())
    display(pheno.y.value_counts())

## Genotype subsetting

In [ ]:
ANALYSES = ['CS_disc', 'SURV']

base_dir = cfg.prep_path(*cfg.prep_parts(ancestry=ANCESTRY, wgs=True))
plink_dir = base_dir / 'PLINK'
plink_dir.mkdir(parents=True, exist_ok=True)

keep_all = prep.keep_samples_across_analyses(base_dir, ANALYSES, ANCESTRY)
keep_file = plink_dir / f'{ANCESTRY}_keep_samples.txt'
keep_all.to_csv(keep_file, sep='\t', index=False)

print(f'{len(keep_all)} unique samples across {ANALYSES}')

In [ ]:
scratch = cfg.TMP_DIR / 'hold_genotypes'
scratch.mkdir(parents=True, exist_ok=True)

script = prep.write_subset_script(
    cfg.WORK_DIR / 'subset_scripts' / ANCESTRY / f'{ANCESTRY}_WGS_subset_plink.sh',
    pfile=cfg.wgs_pfile(ANCESTRY, caller=WGS_CALLER),
    keep_file=keep_file,
    out_prefix=str(scratch / f'chr${{CHROM}}_hy_subsetted_{ANCESTRY}'),
    dest=cfg.prep_url(*cfg.prep_parts(ancestry=ANCESTRY, wgs=True), 'PLINK'),
)
print(f'\ntest on one chromosome first:\n{script} 22')

In [ ]:
for chrom in range(1, 23):
    cfg.shell_do(f'{script} {chrom}', log=True)

## Upload the covariate and phenotype files

In [ ]:
dest = cfg.prep_url(*cfg.prep_parts(ancestry=ANCESTRY, wgs=True))
print(f'gsutil -m cp -r {OUT_DIR} {dest}/')